# 23 · Notebook 21, with the news, market hours and a tidy chart

Everything in notebook 21 — a fitted trendline breaks, the break is **confirmed** by a later
candle, two optional exits — on the same settings, value for value, but for one: the
**trendlines are fitted strict**. Each runs exactly through two real swing extremes and no candle
between its first touch and the moment it is drawn closes through it (notebook 21's fit only
checks the swings, so a wick could cross a "support" by two ATR). Over 19 quarters this changes
the strategy by nothing measurable (-0.02R a trade, t -1.0); it is a cleaner chart, not an edge.
What is new on the chart:

* **the news strip** along the bottom: each release, its forecast, and once out its actual and
  what it meant for the currency (`News:` dropdown, bottom right);
* **market hours** along the top: Sydney, Tokyo, London and New York, each in its own clock
  (`Sessions:` dropdown; `+ opens` adds a faint line at each open);
* **previous day / week / month / year high and low** as dotted steps, labelled PDH / PDL, PWH …
  at the right edge (`Levels:` dropdown, or each one in the legend);
* **support / resistance as bands**, blue while price is above them (support) and orange while it
  is below (resistance), and **gone once price has closed beyond them for about an hour** —
  notebook 21 drew 63 lines on the 5m layer alone over this window, most of them broken;
* **chart and candlestick patterns**, if you name them in §10 of the config.

Settings: **`nb23_config.py`** — §9 news, §10 patterns, §11 market hours & levels, §7 S/R.
Every pattern, layer and setting is listed in **`docs/CHART_GUIDE.md`**.

## 1 · Settings

In [1]:
import sys, os, importlib, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import numpy as np, pandas as pd
from quant.viz import AS_OF_JS

for _p in (os.getcwd(), os.path.join(os.getcwd(), "notebooks"), os.path.abspath("..")):
    if os.path.isfile(os.path.join(_p, "nb23_config.py")) and _p not in sys.path:
        sys.path.insert(0, _p)
import nb23_config as CFG
importlib.reload(CFG)

print(f"{CFG.SYMBOL} {CFG.TF}  {CFG.VIEW_START} -> {CFG.AS_OF or CFG.END}")
_fit = (CFG.TF,) + tuple(t for t in CFG.LINE_TFS if t != CFG.TF)   # the chart's TF is always fitted
print(f"lines fitted on {'+'.join(_fit)}  ·  traded on "
      f"{'+'.join(getattr(CFG, 'TRADE_TFS', None) or _fit)}")
print(f"confirmation: rs = {CFG.TL_CONFIRM_FROM} break bar · {CFG.TL_CONFIRM_N}  "
      f"{'all n' if CFG.TL_CONFIRM_ALL else 'one of n'} · {CFG.TL_CONFIRM_MODE} beyond rs · "
      f"{'own colour' if CFG.TL_CONFIRM_COLOUR else 'any colour'} · "
      f"{'cancels on a close back over rs' if CFG.TL_CONFIRM_CANCEL else 'no cancel'}")
_rules = [(getattr(CFG, d, {}) or {}).get("tag", t) for u, d, t in (
    ("USE_TP_MOVE", "TP_MOVE", "MOVED"), ("USE_FVG_EXIT", "FVG_EXIT", "FVG"),
    ("USE_TP_DECAY", "TP_DECAY", "DECAY"), ("USE_STOP_STEPS", "STOP_STEPS", "STEP"),
    ("USE_STOP_TIME", "STOP_TIME", "TSTOP"), ("USE_HARD_CLOSE", "HARD_CLOSE", "HARD"))
    if getattr(CFG, u, False)]
print(f"exits: target {CFG.TP_RR:g}R  ·  rules on: {', '.join(_rules) or 'none'}")
print(f"news {CFG.NEWS}  ·  levels {CFG.PERIOD_LEVELS}  ·  sessions "
      f"{list(CFG.SESSIONS)}  ·  S/R {CFG.SR_STYLE}")
print(f"settings from {CFG.__file__}")

BTCUSDT 5m  2026-09-01 -> 2026-10-19
lines fitted on 5m+15m+1h+4h  ·  traded on 15m+1h+4h
confirmation: rs = first break bar · {'1h': 3, '15m': 3, '5m': 2, '*': 0}  one of n · close beyond rs · own colour · cancels on a close back over rs
exits: target 3R  ·  rules on: FVG, DECAY
news {'USD': 'medium', 'EUR': 'high', 'GBP': 'high', 'JPY': 'high'}  ·  levels ('D', 'W', 'M', 'Y')  ·  sessions ['Sydney', 'Tokyo', 'London', 'New York']  ·  S/R band
settings from C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\nb23_config.py


## 2 · With the confirmation, and without

`variant()` re-runs the identical study with the rule switched off. Nothing else changes, so every
difference below belongs to the confirmation.

In [2]:
from quant.research import fvg_lab, variant, compare_runs, trade_diff

on  = fvg_lab(CFG)
off = fvg_lab(variant(CFG, TL_CONFIRM_N=0, TL_CONFIRM_CANCEL=False))

compare_runs(confirmation_off=off, confirmation_on=on)

[09:26:28] INFO | quant.data | cache EXTEND binance BTCUSDT 5m -> incremental fetch
[09:26:28] INFO | 01. Parameters: market=spot, symbol=BTCUSDT, interval=5m, start=2026-06-01 00:00:00+00:00, end=2026-10-19 00:00:00+00:00
[09:26:28] INFO | 02. Cache directory: C:\Users\Talal Zahid\Desktop\Talal\Binance\data (cwd=C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks)
[09:26:28] INFO | 03. Main cache file: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\binance_spot_BTCUSDT_5m.parquet
[09:26:28] INFO | 04. Partial checkpoint dir: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\.partials\binance_spot_BTCUSDT_5m
[09:26:28] INFO | 05. Cache: HIT | rows=548,310 (main_rows=548,310, partial_files=0) | range=2021-08-01 00:00:00+00:00 -> 2026-10-18 04:15:00+00:00
[09:26:28] INFO | 06. Detected 3 internal gap(s); will backfill gaps that overlap requested window.
[09:26:28] INFO | 07. Fetch plan 1/1: 2026-10-18 04:20:00+00:00 -> 2026-10-19 00:00:00+00:00
[09:26:28] INFO | 08. Starting fetch range 

BTCUSDT 5m:   0%|          | 0/1 [00:00<?, ?page/s]

[09:26:30] INFO | 09. Completed fetch range 1/1 | rows=2 | total_rows_fetched=2 | requests=1 | retries=0 | elapsed=2.3s
[09:26:31] INFO | 10. Cache consolidated and saved (parquet) | rows=548,312 | range=2021-08-01 00:00:00+00:00 -> 2026-10-18 04:25:00+00:00 | cleared_partial_files=1
[09:26:31] INFO | 11. Fetch summary | new_rows=2 | successful_pages=1 | requests=1 | retries=0 | partial_saves=1 | elapsed=2.8s
[09:26:31] INFO | 12. Returning slice | rows=40,086 | range=2026-06-01 00:00:00+00:00 -> 2026-10-18 04:25:00+00:00
[09:26:31] INFO | quant.data | cache EXTEND binance BTCUSDT 4h -> incremental fetch
[09:26:31] INFO | 01. Parameters: market=spot, symbol=BTCUSDT, interval=4h, start=2026-04-20 00:00:00+00:00, end=2026-10-19 00:00:00+00:00
[09:26:31] INFO | 02. Cache directory: C:\Users\Talal Zahid\Desktop\Talal\Binance\data (cwd=C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks)
[09:26:31] INFO | 03. Main cache file: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\binance_spot_BTC

BTCUSDT 4h:   0%|          | 0/1 [00:00<?, ?page/s]

[09:26:33] INFO | 08. Completed fetch range 1/1 | rows=0 | total_rows_fetched=0 | requests=1 | retries=0 | elapsed=2.1s
[09:26:33] INFO | 09. Cache consolidated and saved (parquet) | rows=4,328 | range=2024-10-27 00:00:00+00:00 -> 2026-10-18 04:00:00+00:00 | cleared_partial_files=0
[09:26:33] INFO | 10. Fetch summary | new_rows=0 | successful_pages=0 | requests=1 | retries=0 | partial_saves=0 | elapsed=2.1s
[09:26:33] INFO | 11. Returning slice | rows=1,088 | range=2026-04-20 00:00:00+00:00 -> 2026-10-18 04:00:00+00:00
[09:26:33] INFO | quant.data | cache EXTEND binance BTCUSDT 1h -> incremental fetch
[09:26:33] INFO | 01. Parameters: market=spot, symbol=BTCUSDT, interval=1h, start=2026-07-29 00:00:00+00:00, end=2026-10-19 00:00:00+00:00
[09:26:33] INFO | 02. Cache directory: C:\Users\Talal Zahid\Desktop\Talal\Binance\data (cwd=C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks)
[09:26:33] INFO | 03. Main cache file: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\binance_spot_BTCUSD

...

BTCUSDT 15m:   0%|          | 0/1 [00:00<?, ?page/s]

[09:27:06] INFO | 09. Completed fetch range 1/1 | rows=0 | total_rows_fetched=0 | requests=1 | retries=0 | elapsed=2.0s
[09:27:06] INFO | 10. Cache consolidated and saved (parquet) | rows=171,085 | range=2021-12-01 00:00:00+00:00 -> 2026-10-18 04:15:00+00:00 | cleared_partial_files=0
[09:27:06] INFO | 11. Fetch summary | new_rows=0 | successful_pages=0 | requests=1 | retries=0 | partial_saves=0 | elapsed=2.2s
[09:27:06] INFO | 12. Returning slice | rows=5,394 | range=2026-08-23 00:00:00+00:00 -> 2026-10-18 04:15:00+00:00
[09:27:06] INFO | quant.data | cache EXTEND binance BTCUSDT 1m -> incremental fetch
[09:27:06] INFO | 01. Parameters: market=spot, symbol=BTCUSDT, interval=1m, start=2026-08-31 00:00:00+00:00, end=2026-10-19 00:00:00+00:00
[09:27:06] INFO | 02. Cache directory: C:\Users\Talal Zahid\Desktop\Talal\Binance\data (cwd=C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks)
[09:27:06] INFO | 03. Main cache file: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\binance_spot_BTCU

BTCUSDT 1m:   0%|          | 0/2 [00:00<?, ?page/s]

[09:27:08] INFO | 09. Completed fetch range 1/1 | rows=1 | total_rows_fetched=1 | requests=1 | retries=0 | elapsed=2.4s
[09:27:09] INFO | 10. Cache consolidated and saved (parquet) | rows=1,552,548 | range=2022-01-01 00:00:00+00:00 -> 2026-10-18 04:27:00+00:00 | cleared_partial_files=1
[09:27:09] INFO | 11. Fetch summary | new_rows=1 | successful_pages=1 | requests=1 | retries=0 | partial_saves=1 | elapsed=3.7s
[09:27:09] INFO | 12. Returning slice | rows=69,388 | range=2026-08-31 00:00:00+00:00 -> 2026-10-18 04:27:00+00:00
[09:27:11] INFO | quant.engine | backtest done | bars=40,086 trades=4 final_cash=10037.73 return=0.38% dd=2.65% | 4.0 ms
[09:27:11] INFO | quant.engine | backtest done | bars=40,086 trades=8 final_cash=9927.92 return=-0.72% dd=5.54% | 4.0 ms


,run,trades,total R,wins,R / trade
0,confirmation off,12,-2.04,4,-0.170
1,confirmation on,8,0.94,3,0.117


## 3 · Which trades it changed

A confirmation can only move an entry later or remove it, so read the rows that appear in one
column and not the other. `1h` in `trigger` is the timeframe this rule is set for.

In [3]:
trade_diff(off=off, on=on)

,side off,trigger off,R off,close_reason off,side on,trigger on,R on,close_reason on
entry,,,,,,,,
2026-09-01 05:45,long,15m,-1.237365,stop_loss,NaN,NaN,NaN,NaN
2026-09-01 05:50,NaN,NaN,NaN,NaN,long,15m,-1.239834,stop_loss
2026-09-05 11:55,short,15m,-2.193337,stop_loss,NaN,NaN,NaN,NaN
2026-09-08 13:35,short,15m,-1.190679,stop_loss,NaN,NaN,NaN,NaN
2026-09-08 13:40,NaN,NaN,NaN,NaN,short,15m,-1.182055,stop_loss
2026-09-18 01:40,long,15m,1.327373,fvg_exit,NaN,NaN,NaN,NaN
2026-09-18 01:45,NaN,NaN,NaN,NaN,long,15m,2.727029,take_profit
2026-09-24 02:05,short,15m,-1.206531,stop_loss,NaN,NaN,NaN,NaN
2026-09-24 03:50,NaN,NaN,NaN,NaN,short,15m,-1.237469,stop_loss


## 4 · The setups that did NOT trade

A filter can only be judged against what it removed — and the chart shows only what survived, so a
rule that deletes a good setup looks identical to one that deletes nothing. Every break that was
turned down, with the reason:

* **entry rule** — the `ENTRY_RULES` sentence was false on the break bar
* **not confirmed** — the window expired with no qualifying candle
* **cancelled** — price closed back over rs inside the window
* **already open** — the signal was fine, but a position was already running

On the chart they are one legend entry per reason, **all starting off**, so the view opens on price
and trades exactly as before. Click one in the legend to bring it in.

In [4]:
on.rejected.groupby(["tf", "reason"]).size().rename("breaks").reset_index()

,tf,reason,breaks
0,15m,already open,3
1,15m,entry rule,1
2,15m,not confirmed,8
3,1h,not confirmed,3


## 5 · Two optional exits

Both leave the entry alone and change only how a trade ends.

* **Strategy 1, tag `MOVED`** (`USE_TP_MOVE`) — price got `near` of the way to the target and a
  close then gave `back` of it back, or the trade has been open `after_bars`: the target moves to
  `to` of its distance. Judged on the close, used from the next bar.
* **Strategy 2, tag `FVG`** (`USE_FVG_EXIT`) — a live gap from `tfs` sits within `near_r` R of the
  target when the trade opens: price filling it closes the trade (at its `touch`, `ce` or `full`).

§3 of the config has four more, each off until switched on and each with its own tag:
**`DECAY`** (the target shrinks every N hours open), **`STEP`** and **`TSTOP`** (the stop moves
toward profit at a share of the way to the target, or after N hours), and **`HARD`** (out after N
hours). Hours can be set per line timeframe; a trade uses the settings of the coarsest line that
fired it. Every tag is written on the chart's exit marker — `TP·DECAY`, `SL·TSTOP`, `HARD` — beside
the plain `TP`, `SL` and `BE`.

Each arm of `COMPARE` re-runs the notebook with only its own change, so the columns differ by
exactly what they name. Both rules were **designed from one trade** in this window, which is why it
looks good here — the verdict over two years is in `docs/WHAT_WORKS.md`.

In [5]:
from quant.research import run_arms

arms = run_arms(CFG)          # one run per arm of COMPARE
arms.table()

[09:27:13] INFO | quant.data | cache EXTEND binance BTCUSDT 5m -> incremental fetch
[09:27:13] INFO | 01. Parameters: market=spot, symbol=BTCUSDT, interval=5m, start=2026-06-01 00:00:00+00:00, end=2026-10-19 00:00:00+00:00
[09:27:13] INFO | 02. Cache directory: C:\Users\Talal Zahid\Desktop\Talal\Binance\data (cwd=C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks)
[09:27:13] INFO | 03. Main cache file: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\binance_spot_BTCUSDT_5m.parquet
[09:27:13] INFO | 04. Partial checkpoint dir: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\.partials\binance_spot_BTCUSDT_5m
[09:27:13] INFO | 05. Cache: HIT | rows=548,312 (main_rows=548,312, partial_files=0) | range=2021-08-01 00:00:00+00:00 -> 2026-10-18 04:25:00+00:00
[09:27:13] INFO | 06. Detected 3 internal gap(s); will backfill gaps that overlap requested window.
[09:27:13] INFO | 07. Fetch plan 1/1: 2026-10-18 04:30:00+00:00 -> 2026-10-19 00:00:00+00:00
[09:27:13] INFO | 08. Starting fetch range 

BTCUSDT 5m:   0%|          | 0/1 [00:00<?, ?page/s]

[09:27:16] INFO | 09. Completed fetch range 1/1 | rows=0 | total_rows_fetched=0 | requests=1 | retries=0 | elapsed=2.5s
[09:27:16] INFO | 10. Cache consolidated and saved (parquet) | rows=548,312 | range=2021-08-01 00:00:00+00:00 -> 2026-10-18 04:25:00+00:00 | cleared_partial_files=0
[09:27:16] INFO | 11. Fetch summary | new_rows=0 | successful_pages=0 | requests=1 | retries=0 | partial_saves=0 | elapsed=2.9s
[09:27:16] INFO | 12. Returning slice | rows=40,086 | range=2026-06-01 00:00:00+00:00 -> 2026-10-18 04:25:00+00:00
[09:27:16] INFO | quant.data | cache EXTEND binance BTCUSDT 4h -> incremental fetch
[09:27:16] INFO | 01. Parameters: market=spot, symbol=BTCUSDT, interval=4h, start=2026-04-20 00:00:00+00:00, end=2026-10-19 00:00:00+00:00
[09:27:16] INFO | 02. Cache directory: C:\Users\Talal Zahid\Desktop\Talal\Binance\data (cwd=C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks)
[09:27:16] INFO | 03. Main cache file: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\binance_spot_BTC

BTCUSDT 4h:   0%|          | 0/1 [00:00<?, ?page/s]

[09:27:18] INFO | 08. Completed fetch range 1/1 | rows=0 | total_rows_fetched=0 | requests=1 | retries=0 | elapsed=1.9s
[09:27:18] INFO | 09. Cache consolidated and saved (parquet) | rows=4,328 | range=2024-10-27 00:00:00+00:00 -> 2026-10-18 04:00:00+00:00 | cleared_partial_files=0
[09:27:18] INFO | 10. Fetch summary | new_rows=0 | successful_pages=0 | requests=1 | retries=0 | partial_saves=0 | elapsed=1.9s
[09:27:18] INFO | 11. Returning slice | rows=1,088 | range=2026-04-20 00:00:00+00:00 -> 2026-10-18 04:00:00+00:00
[09:27:18] INFO | quant.data | cache EXTEND binance BTCUSDT 1h -> incremental fetch
[09:27:18] INFO | 01. Parameters: market=spot, symbol=BTCUSDT, interval=1h, start=2026-07-29 00:00:00+00:00, end=2026-10-19 00:00:00+00:00
[09:27:18] INFO | 02. Cache directory: C:\Users\Talal Zahid\Desktop\Talal\Binance\data (cwd=C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks)
[09:27:18] INFO | 03. Main cache file: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\binance_spot_BTCUSD

...

BTCUSDT 15m:   0%|          | 0/1 [00:00<?, ?page/s]

[09:27:29] INFO | 09. Completed fetch range 1/1 | rows=0 | total_rows_fetched=0 | requests=1 | retries=0 | elapsed=2.2s
[09:27:29] INFO | 10. Cache consolidated and saved (parquet) | rows=171,085 | range=2021-12-01 00:00:00+00:00 -> 2026-10-18 04:15:00+00:00 | cleared_partial_files=0
[09:27:29] INFO | 11. Fetch summary | new_rows=0 | successful_pages=0 | requests=1 | retries=0 | partial_saves=0 | elapsed=2.5s
[09:27:29] INFO | 12. Returning slice | rows=5,394 | range=2026-08-23 00:00:00+00:00 -> 2026-10-18 04:15:00+00:00
[09:27:29] INFO | quant.data | cache EXTEND binance BTCUSDT 1m -> incremental fetch
[09:27:29] INFO | 01. Parameters: market=spot, symbol=BTCUSDT, interval=1m, start=2026-08-31 00:00:00+00:00, end=2026-10-19 00:00:00+00:00
[09:27:29] INFO | 02. Cache directory: C:\Users\Talal Zahid\Desktop\Talal\Binance\data (cwd=C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks)
[09:27:29] INFO | 03. Main cache file: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\binance_spot_BTCU

BTCUSDT 1m:   0%|          | 0/2 [00:00<?, ?page/s]

[09:27:32] INFO | 09. Completed fetch range 1/1 | rows=0 | total_rows_fetched=0 | requests=1 | retries=0 | elapsed=2.3s
[09:27:33] INFO | 10. Cache consolidated and saved (parquet) | rows=1,552,548 | range=2022-01-01 00:00:00+00:00 -> 2026-10-18 04:27:00+00:00 | cleared_partial_files=0
[09:27:33] INFO | 11. Fetch summary | new_rows=0 | successful_pages=0 | requests=1 | retries=0 | partial_saves=0 | elapsed=3.7s
[09:27:33] INFO | 12. Returning slice | rows=69,388 | range=2026-08-31 00:00:00+00:00 -> 2026-10-18 04:27:00+00:00
[09:27:35] INFO | quant.engine | backtest done | bars=40,086 trades=3 final_cash=9971.71 return=-0.28% dd=1.68% | 15.7 ms
[09:27:35] INFO | quant.engine | backtest done | bars=40,086 trades=5 final_cash=9921.52 return=-0.78% dd=3.42% | 5.2 ms
[09:27:35] INFO | quant.engine | backtest done | bars=40,086 trades=3 final_cash=9971.71 return=-0.28% dd=1.68% | 4.2 ms
[09:27:36] INFO | quant.engine | backtest done | bars=40,086 trades=5 final_cash=9921.52 return=-0.78% dd=

,run,trades,total R,wins,R / trade,stop_loss,take_profit,tp_decayed,tp_moved
0,neither,8,0.94,3,0.117,5,2,1,0
1,move target,8,0.94,3,0.117,5,2,0,1
2,fvg exit,8,0.94,3,0.117,5,2,1,0
3,both,8,0.94,3,0.117,5,2,0,1


In [6]:
arms.diff()

,side,trigger,neither R,neither exit,move target R,move target exit,fvg exit R,fvg exit exit,both R,both exit
entry,,,,,,,,,,
2026-09-01 05:50,long,15m,-1.24,stop,-1.24,stop,-1.24,stop,-1.24,stop
2026-09-08 13:40,short,15m,-1.18,stop,-1.18,stop,-1.18,stop,-1.18,stop
2026-09-18 01:45,long,15m,2.73,target,2.73,target,2.73,target,2.73,target
2026-09-24 03:50,short,15m,-1.24,stop,-1.24,stop,-1.24,stop,-1.24,stop
2026-09-24 08:40,short,15m,-1.16,stop,-1.16,stop,-1.16,stop,-1.16,stop
2026-09-30 12:35,long,15m,-1.12,stop,-1.12,stop,-1.12,stop,-1.12,stop
2026-10-07 00:25,short,15m,2.43,target,2.43,target,2.43,target,2.43,target
2026-10-07 02:00,short,1h,1.72,tp_decayed,1.72,moved target,1.72,tp_decayed,1.72,moved target


## 6 · The chart

Drawn as the settings stand. **Every exit marker carries its tag** (`EXIT_LABELS`): `TP` target,
`SL` stop, `BE` stop at breakeven, `TP·MOVED` / `TP·FVG` / `TP·DECAY` for the exit rules.

| on the chart | where it is set | switch it on the chart |
|---|---|---|
| news strip (bottom) | §9 `NEWS`, `NEWS_SHOW` | `News:` dropdown, bottom right |
| previous D / W / M / Y high & low | §11 `PERIOD_LEVELS`, `PERIOD_LEVELS_ON`, `LEVELS_TZ` | `Levels:` dropdown, or the legend |
| market sessions (top) | §11 `SESSIONS`, `SESSION_OPEN_LINES` | `Sessions:` dropdown, or the legend |
| S/R bands, blue support / orange resistance | §7 `SR_STYLE`, `SR_ROLES`, `SR_BREAK_BARS` | the `S/R:` button row, top left |
| chart patterns (shapes) | **§10 `CHART_PATTERNS`** (names in docs/CHART_GUIDE.md) | the pattern menus, top centre |
| candlestick patterns (triangles) | **§10 `CANDLE_PATTERNS`** | the legend |
| fair value gaps | §6 | the `FVGs:` button row |

**Rewind** (bottom left) takes the levels, bands, gaps, lines and trades back to what was known at
the candle you click. The news schedule and the session hours stay, because they were known in
advance; the news actuals disappear until they came out.

In [7]:
fig = on.figure()
# fig

[09:27:39] INFO | quant.data | cache EXTEND binance BTCUSDT 1h -> incremental fetch
[09:27:39] INFO | 01. Parameters: market=spot, symbol=BTCUSDT, interval=1h, start=2024-12-31 00:00:00+00:00, end=2026-10-19 00:00:00+00:00
[09:27:39] INFO | 02. Cache directory: C:\Users\Talal Zahid\Desktop\Talal\Binance\data (cwd=C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks)
[09:27:39] INFO | 03. Main cache file: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\binance_spot_BTCUSDT_1h.parquet
[09:27:39] INFO | 04. Partial checkpoint dir: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\.partials\binance_spot_BTCUSDT_1h
[09:27:39] INFO | 05. Cache: HIT | rows=45,242 (main_rows=45,242, partial_files=0) | range=2021-08-20 00:00:00+00:00 -> 2026-10-18 04:00:00+00:00
[09:27:39] INFO | 06. Detected 2 internal gap(s); will backfill gaps that overlap requested window.
[09:27:39] INFO | 07. Fetch plan 1/1: 2026-10-18 05:00:00+00:00 -> 2026-10-19 00:00:00+00:00
[09:27:39] INFO | 08. Starting fetch range 1/

BTCUSDT 1h:   0%|          | 0/1 [00:00<?, ?page/s]

[09:27:41] INFO | 09. Completed fetch range 1/1 | rows=0 | total_rows_fetched=0 | requests=1 | retries=0 | elapsed=2.0s
[09:27:41] INFO | 10. Cache consolidated and saved (parquet) | rows=45,242 | range=2021-08-20 00:00:00+00:00 -> 2026-10-18 04:00:00+00:00 | cleared_partial_files=0
[09:27:41] INFO | 11. Fetch summary | new_rows=0 | successful_pages=0 | requests=1 | retries=0 | partial_saves=0 | elapsed=2.1s
[09:27:41] INFO | 12. Returning slice | rows=15,749 | range=2024-12-31 00:00:00+00:00 -> 2026-10-18 04:00:00+00:00


In [8]:
from quant.viz import open_in_browser

open_in_browser(fig, "nb23.html", extra_js=AS_OF_JS)

WindowsPath('C:/Users/Talal Zahid/Desktop/Talal/Binance/notebooks/nb23.html')

---
### What to change

| want | setting |
|---|---|
| any bar of the break may be rs | `TL_CONFIRM_FROM = "every"` |
| more or less grace after rs | `TL_CONFIRM_N = {"1h": 3, "*": 0}` |
| every candle must hold, not just one | `TL_CONFIRM_ALL = True` |
| the whole candle beyond rs, not its close | `TL_CONFIRM_MODE = "full"` |
| a bounce should not void the setup | `TL_CONFIRM_CANCEL = False` |
| confirm 15m breaks too | add `"15m"` to `TL_CONFIRM_N` |
| stop trading one timeframe's lines (still drawn) | drop it from `TRADE_TFS`, e.g. `("15m", "1h")` |
| rename what the chart writes | `tag=` in the rule's settings |
| shrink the target with time / move the stop / close after N hours | `USE_TP_DECAY` / `USE_STOP_STEPS`, `USE_STOP_TIME` / `USE_HARD_CLOSE` (§3) |
| no labels on the exit markers | `EXIT_LABELS = False` (§8) |
| move the target after a near miss | `USE_TP_MOVE = True`, tune `TP_MOVE` |
| exit when a gap near the target fills | `USE_FVG_EXIT = True`, tune `FVG_EXIT` |
| a different A/B table | edit `COMPARE` |
| draw chart patterns | `CHART_PATTERNS = ["double_top", "rising_wedge"]` or `"all"` (§10) |
| mark candlestick patterns | `CANDLE_PATTERNS = ["engulfing_bull", "hammer"]` (§10) |
| open with the monthly levels on | `PERIOD_LEVELS_ON = ("D", "W", "M")` (§11) |
| a New-York-close trading day for the levels | `LEVELS_TZ = "America/New_York"` (§11) |
| the stock-market open for Nasdaq | `"New York": ("America/New_York", "09:30", "16:00")` (§11) |
| notebook 21's looser trendline fit | `fit="ransac", line_tol_atr=0.6` in `TRENDLINE` (§2) |
| S/R as lines again, or never retired | `SR_STYLE = "line"`, `SR_BREAK_BARS = None` (§7) |
| the news from medium releases up | `NEWS_SHOW = "medium"` (§9) |

Eight trades over nine days is not evidence either way. `experiments/trendline_wait_search.py`
scores this across six windows against a chance baseline; that is what settles it.